## Manage mlflow experiments

In [2]:
import mlflow
import pandas as pd
from mlflow.tracking import MlflowClient

c:\Users\fkeyu\Documents\MyCode\Python\space-titanic-problem\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Getting the test data

In [3]:
# ensured that raw test.csv is preprocessd and is stored in relevant diectory

X_test = pd.read_csv('../data/pre_processed/test.csv')
raw_X_test = pd.read_csv('../data/raw/test.csv')

### Load mlflow model for prediction

In [4]:
# configuring the sqlite db path since it is in the parent folder
mlflow.set_tracking_uri('sqlite:///../mlflow.db')

### Checking available run results

In [5]:
# connecting to tracking backend (ensure tracking uri is configured correctly)
client = MlflowClient()

# list all runs for a given experiment (replace with your experiment_id)
experiment_id = "1"   # default experiment is usually "0", in this case we hv logged runs at id 1
runs = client.search_runs(experiment_ids=[experiment_id])

# loop through runs and print params + metrics
for run in runs:
    print("Run ID:", run.info.run_id)
    print("Params:", run.data.params)
    print("Metrics:", run.data.metrics)
    print("-" * 40)

Run ID: ff393746f13a4bceb859c16d4a78567d
Params: {'max_iter': '10000', 'current_threshold': '0.5', 'optimal_threshold': '0.558582257325105'}
Metrics: {'accuracy_score': 0.781441717791411, 'auc_score': 0.8602467190395516}
----------------------------------------
Run ID: da75a3a73d4c40bda8239f8c55477d17
Params: {'max_iter': '10000', 'current_threshold': '0.5', 'optimal_threshold': '0.558582257325105'}
Metrics: {'accuracy_score': 0.781441717791411, 'auc_score': 0.8602467190395516}
----------------------------------------
Run ID: 509cd4f8d5354b12ba3e38ceeedb1f27
Params: {'max_iter': '10000', 'current_threshold': '0.5', 'optimal_threshold': '0.558582257325105'}
Metrics: {'accuracy_score': 0.781441717791411, 'auc_score': 0.8602467190395516}
----------------------------------------
Run ID: 18b6b7dab0e4477692b020d5fbed2688
Params: {'max_iter': '10000', 'current_threshold': '0.5', 'optimal_threshold': '0.5580839693983952'}
Metrics: {'accuracy_score': 0.781058282208589, 'auc_score': 0.8602920609

### Loading a particular model

In [6]:
# loading the model using mlflow run id
run_id = "18b6b7dab0e4477692b020d5fbed2688"
model = mlflow.sklearn.load_model(f"runs:/{run_id}/model")

### Register best model as 'Champion' Model

In [18]:
# find best run by accuracy_score metric
best_run = max(runs, key=lambda r: r.data.metrics.get("accuracy_score", 0))
best_run_id = best_run.info.run_id

In [16]:
mlflow.register_model(f"runs:/{best_run_id}/model", "Champion")

Successfully registered model 'Champion'.
2026/08/06 23:09:44 WARNING mlflow.tracking._model_registry.fluent: Run with id ff393746f13a4bceb859c16d4a78567d has no artifacts at artifact path 'model', registering model based on models:/m-4e2c54ecaa3c4706a0ca6e79bb50f931 instead
Created version '1' of model 'Champion'.


<ModelVersion: aliases=[], creation_timestamp=1786037984167, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1786037984167, metrics=None, model_id=None, name='Champion', params=None, run_id='ff393746f13a4bceb859c16d4a78567d', run_link=None, source='models:/m-4e2c54ecaa3c4706a0ca6e79bb50f931', status='READY', status_message=None, tags={}, user_id=None, version=1, workspace='default'>

### Making prediction using this model

In [17]:
run_id = best_run_id
final_model = mlflow.sklearn.load_model(f"runs:/{run_id}/model")
current_threshold = 0.5584083119472486


# getting prediction using the loaded model
y_prob = final_model.predict_proba(X_test)
y_pred = y_prob[:, 1] >= current_threshold

In [10]:
# converting results to a df, and then a csv for submission
final_submission_df = pd.DataFrame({
    "PassengerId": raw_X_test.PassengerId.values,
    "Transported": y_pred.flatten()
})

final_submission_df.to_csv(f"../data/output/run_{run_id}_predictions.csv", index=False)

### Way to get the latest run with non-null value of metrics

In [65]:
latest_run = mlflow.search_runs(
    experiment_ids=[1],
    filter_string="metric.accuracy_score > 0",
    order_by=["start_time DESC"],   # latest run first
    max_results=1
)

latest_run_id = latest_run.run_id[0]

mlflow.pyfunc.load_model(f"runs:/{latest_run_id}/model")

mlflow.pyfunc.loaded_model:
  artifact_path: file:C:/Users/fkeyu/Documents/MyCode/Python/space-titanic-problem/mlruns/1/models/m-4e2c54ecaa3c4706a0ca6e79bb50f931/artifacts
  flavor: mlflow.sklearn
  run_id: ff393746f13a4bceb859c16d4a78567d